<center><img src="../../image/dice.png" alt="DICE" style="width:300px;"/></center>

# 8. 四个堆叠方块归位

@－－－－DICE－－－－<br>
@ 文件名：08.四个堆叠方块归位.ipynb <br>
@ 版本：V1.0 <br>
@ 说明：中央四层堆叠方块从上到下依次抓取，分别归位到 A/B/C/D 四个区域<br>

> 本 Notebook 复用 `2.机械臂实验` 中的 `Arm5DoFUServo`、`Arm5DoFKinematic`、`icp_solver_svd` 和九点标定流程；
> 标定数据采用 2026-10 的新九点实测值，不再沿用旧工作台参数。


## 1. 运行模式与依赖

- `RUN_MODE = "sim"`：离线运动学与流程仿真，不连接机械臂，可直接从头运行到结尾。
- `RUN_MODE = "real"`：连接真机。首次运行前必须确认工作台区域无人、急停按钮可用，并建议先只跑一轮。
- 仿真模式用于检查：九点标定、工作台坐标变换、四层抓取高度、A/B/C/D 目标位姿、逆运动学可达性、动作顺序和路径安全高度。
- 真机模式复用原实验的串口舵机 SDK，`gripper_open/close`、`set_tool_pose` 等均会等待舵机到位。


In [ ]:
%matplotlib inline
import os
import sys
import math
import time
import json
import csv
from dataclasses import dataclass, field, asdict
from datetime import datetime

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, FancyArrowPatch, Circle
from matplotlib.lines import Line2D
from IPython.display import display, HTML

# ===================== 运行模式 =====================
RUN_MODE = "sim"          # "sim" 或 "real"
REAL_CONFIRM = False      # 真机运行前改为 True
REAL_ROUNDS = 1           # 真机建议先跑 1 轮

# 中文显示
plt.rcParams["font.sans-serif"] = [
    "SimHei", "Microsoft YaHei", "WenQuanYi Micro Hei",
    "Noto Sans CJK SC", "DejaVu Sans"
]
plt.rcParams["axes.unicode_minus"] = False
np.set_printoptions(precision=4, suppress=True)

RESULT_DIR = "results"
IMAGE_DIR = "image"
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)
print("RUN_MODE =", RUN_MODE)
print("工作目录 =", os.getcwd())


## 2. 九点标定与工作台坐标变换

九点标定板尺寸按 `config/handeye_9points.yaml` 取 **宽 133 mm × 高 100 mm**。  
工作台坐标系原点位于 P4，X 轴指向 P1/P2 一侧，Y 轴指向 P0/P6 一侧。  
先把九点 Z 值按“半径—Z 误差”做一次线性平面补偿，再用 SVD-ICP 求 `T_arm2ws`。


In [ ]:
from icp import icp_solver_svd
from transform import Transform
from arm5dof_kinematic import Arm5DoFKinematic
from config import ARM_POSE_DICT, PITCH_PANEL_A, PITCH_PANEL_B, PITCH_PANEL_C

# ===================== 2026-10 新九点实测数据（机械臂坐标，mm） =====================
ARM_9P_MEASURED = np.array([
    [236.689,  48.427,  -8.298],
    [230.298, -12.795, -14.189],
    [229.308, -77.589,  -0.554],
    [190.426,  53.205,  -3.096],
    [194.361,  -7.849,  -4.952],
    [191.712, -66.670,  -2.420],
    [150.223,  52.242,   0.724],
    [142.433,  -8.634,  -1.878],
    [139.329, -63.991,  -0.532],
], dtype=np.float64)

BOARD_WIDTH_MM = 133.0
BOARD_HEIGHT_MM = 100.0
X0 = BOARD_HEIGHT_MM / 2.0      # P0/P1/P2 的 X
Y0 = BOARD_WIDTH_MM / 2.0       # P0/P3/P6 的 Y

# 九点标定板在工作台坐标系下的理论坐标（原点 P4）
WS_9P = np.float64([
    [ X0,  Y0, 0.0],   # P0
    [ X0,  0.0, 0.0],   # P1
    [ X0, -Y0, 0.0],   # P2
    [0.0,  Y0, 0.0],   # P3
    [0.0,  0.0, 0.0],   # P4
    [0.0, -Y0, 0.0],   # P5
    [-X0,  Y0, 0.0],   # P6
    [-X0,  0.0, 0.0],   # P7
    [-X0, -Y0, 0.0],   # P8
])

# 关节臂的 Z 值与半径相关，先拟合 z_error(r)，再把九点 Z 拉平
Z_MEAN = float(np.mean(ARM_9P_MEASURED[:, 2]))
RADII = np.hypot(ARM_9P_MEASURED[:, 0], ARM_9P_MEASURED[:, 1])
Z_ERROR_POLY = np.polyfit(RADII, ARM_9P_MEASURED[:, 2] - Z_MEAN, 1)  # [k, b]
ARM_9P_FLAT = ARM_9P_MEASURED.copy()
ARM_9P_FLAT[:, 2] = Z_MEAN

# 求机械臂坐标系 -> 工作台坐标系的齐次变换
R_ARM2WS, t_ARM2WS = icp_solver_svd(ARM_9P_FLAT, WS_9P)
T_ARM2WS = np.eye(4)
T_ARM2WS[:3, :3] = R_ARM2WS
T_ARM2WS[:3, 3] = np.asarray(t_ARM2WS).reshape(-1)
T_WS2ARM = Transform.inverse(T_ARM2WS)

# 标定残差
ARM_PRED = (T_ARM2WS @ np.c_[WS_9P, np.ones(len(WS_9P))].T).T[:, :3]
RESIDUAL = ARM_PRED - ARM_9P_FLAT
RESIDUAL_NORM = np.linalg.norm(RESIDUAL, axis=1)

print("Z_MEAN = {:.4f} mm".format(Z_MEAN))
print("z_error(r) = {:.4f} * r + {:.4f}".format(*Z_ERROR_POLY))
print("T_arm2ws =")
print(np.array2string(T_ARM2WS, precision=4, suppress_small=True))
print("九点残差：max = {:.3f} mm, mean = {:.3f} mm".format(
    RESIDUAL_NORM.max(), RESIDUAL_NORM.mean()))

try:
    T_SAVED = np.loadtxt("config/T_arm2ws.txt", delimiter=",")
    print("与 config/T_arm2ws.txt 的最大绝对差异 = {:.6f}".format(
        np.abs(T_SAVED - T_ARM2WS).max()))
except Exception as exc:
    print("读取 config/T_arm2ws.txt 失败（不影响本次计算）：", exc)


## 3. 任务参数与四层位姿表

- 方块高度：`28 mm`（来自 `config/object.yaml`，本 Notebook 设为可调参数）。
- 中央堆叠：4 块，自下而上第 1～4 层，抓取顺序为 **第 4 层 → 第 3 层 → 第 2 层 → 第 1 层**。
- A/B/C/D 四个目标区取工作台坐标 `(±55, ±80) mm`，与 `07.机械臂抓取工作台物块测试.ipynb` 的四周区域一致。
- 安全高度 `200 mm`；抓取上方点比抓取点高 `60 mm`。


In [ ]:
# ===================== 任务参数 =====================
# 方块高度优先从 config/object.yaml 读取；仿真环境没有 PyYAML 时回退到 28 mm
try:
    import yaml
    with open("config/object.yaml", "r", encoding="utf-8") as _f:
        CUBE_SIZE_MM = float(yaml.safe_load(_f).get("height", 28.0))
except Exception:
    CUBE_SIZE_MM = 28.0

STACK_XY = (0.0, 0.0)

TARGETS = {
    "A": ( 55.0,  80.0),   # 右前 / P0 外侧
    "B": ( 55.0, -80.0),   # 右后 / P2 外侧
    "C": (-55.0, -80.0),   # 左后 / P8 外侧
    "D": (-55.0,  80.0),   # 左前 / P6 外侧
}

SAFE_Z_MM = 200.0
APPROACH_CLEARANCE_MM = 60.0
PLACE_Z_MM = CUBE_SIZE_MM / 2.0
GRIP_Z_OFFSET_MM = 0.0
OPEN_ANGLE_RAD = math.radians(30.0)

@dataclass
class TaskConfig:
    cube_size: float = CUBE_SIZE_MM
    stack_xy: tuple = STACK_XY
    targets: dict = field(default_factory=lambda: dict(TARGETS))
    safe_z: float = SAFE_Z_MM
    approach_clearance: float = APPROACH_CLEARANCE_MM
    place_z: float = PLACE_Z_MM
    grip_z_offset: float = GRIP_Z_OFFSET_MM
    open_angle: float = OPEN_ANGLE_RAD
    pick_steps: int = 5
    place_steps: int = 8
    transfer_steps: int = 6
    settle_s: float = 0.25

    def pick_z(self, layer: int) -> float:
        return self.cube_size * (layer + 0.5) + self.grip_z_offset

    def approach_z(self, layer: int) -> float:
        return self.pick_z(layer) + self.approach_clearance

    def plan(self):
        return [(3, "A"), (2, "B"), (1, "C"), (0, "D")]

cfg = TaskConfig()

print("中央堆叠中心：", cfg.stack_xy)
print("方块高度：{:.1f} mm，堆叠总高：{:.1f} mm".format(
    cfg.cube_size, cfg.cube_size * 4))
print()
print("{:<10} {:<8} {:>10} {:>12} {:>10} {:>10}".format(
    "方块", "目标区", "抓取Z/mm", "上方点Z/mm", "放置Z/mm", "目标XY/mm"))
for layer, target in cfg.plan():
    print("{:<10} {:<8} {:>10.1f} {:>12.1f} {:>10.1f} {:>10}".format(
        "第{}层".format(layer + 1), target,
        cfg.pick_z(layer), cfg.approach_z(layer), cfg.place_z,
        str(cfg.targets[target])))


## 4. 仿真机械臂（离线验证）

`VirtualArm` 不连接串口，只用 `Arm5DoFKinematic` 做逆解、正解和关节限位检查，并复现真机 SDK 的以下行为：

1. `set_tool_pose()` 先做 `z_error(r)` 补偿，再自动生成俯仰角；
2. 逆解无解时返回 `False`；
3. `gripper_open/close` 记录夹爪状态；
4. `home()` 回到 `ARM_POSE_DICT["home"]`。

因此同一套 `StackUnstackTask` 可以同时跑仿真和真机。


In [ ]:
class VirtualArm:
    """只做运动学验证的虚拟机械臂，接口与 Arm5DoFUServo 保持兼容。"""
    is_virtual = True

    def __init__(self, T_arm2ws, z_error_polyfit):
        self.T_arm2ws = np.asarray(T_arm2ws, dtype=np.float64)
        self.z_error_polyfit = np.asarray(z_error_polyfit, dtype=np.float64)
        self.kin = Arm5DoFKinematic()
        self.joints = np.array(self.kin.JOINT_ANGLE_DEFAULT[:5], dtype=np.float64)
        self.pose = np.array(self.kin.forward_kinematic_v2(self.joints), dtype=np.float64)
        self.gripper = "open"
        self.command_count = 0
        self.ik_fail = 0

    def _adjust(self, p):
        x, y, z = p
        r = math.hypot(x, y)
        k, b = self.z_error_polyfit
        return [x, y, z + k * r + b]

    def _auto_pitch(self, p):
        x, y, z = p
        return PITCH_PANEL_A * math.hypot(x, y) + PITCH_PANEL_B * z + PITCH_PANEL_C

    def set_tool_pose(self, tool_posi=None, tool_pitch=None, tool_roll=np.pi,
                      pose_name=None, T=1.0, **kwargs):
        if pose_name is not None:
            x, y, z, tool_pitch, tool_roll = ARM_POSE_DICT[pose_name]
            tool_posi = [x, y, z]
        p = self._adjust(tool_posi)
        if tool_pitch is None:
            tool_pitch = self._auto_pitch(p)
        sols = self.kin.inverse_kinematic(
            p[0], p[1], p[2], tool_pitch, tool_roll,
            last_joint_angle=self.joints)
        if len(sols) == 0:
            self.ik_fail += 1
            return False
        sols = np.asarray(sols, dtype=np.float64)
        idx = int(np.argmin(np.linalg.norm(sols - self.joints, axis=1)))
        self.joints = sols[idx]
        self.pose = np.asarray(
            self.kin.forward_kinematic_v2(self.joints), dtype=np.float64)
        self.command_count += 1
        return True

    def home(self):
        return self.set_tool_pose(pose_name="home", T=2.0)

    def gripper_open(self, angle=np.pi / 6, T=0.5, **kwargs):
        self.gripper = "open"
        return True

    def gripper_close(self, T=0.5, **kwargs):
        self.gripper = "closed"
        return True

    def set_damping(self, power=0):
        pass

    def get_tool_pose(self):
        return self.pose.copy()

print("VirtualArm 定义完成")


## 5. Pick & Place 控制器

动作流程固定为：

`Approach（抓取上方点） → Pick（下降到抓取点、夹紧） → Lift（抬升） → Transfer（安全高度横移） → Place（分段下降、松开） → Retreat（抬升退出）`

函数封装：

- `pick(layer)`：完成一层抓取，只允许抓当前最上层；
- `place(target)`：完成一个目标区放置；
- `move_z()` / `move_xy()`：把大幅动作拆成小步，避免关节插补造成的大幅斜向扫掠；
- `run()`：按 `3→A, 2→B, 1→C, 0→D` 顺序执行，并记录每一块的结果。


In [ ]:
class SafetyError(RuntimeError):
    pass

class StackUnstackTask:
    def __init__(self, arm, T_arm2ws, config):
        self.arm = arm
        self.T_arm2ws = np.asarray(T_arm2ws, dtype=np.float64)
        self.cfg = config
        self.remaining = [0, 1, 2, 3]   # 自下而上，3 为顶层
        self.current_ws = None
        self.offset_pick = (0.0, 0.0)
        self.offset_place = (0.0, 0.0)
        self.ws_path = []
        self.block_log = []
        self.is_virtual = bool(getattr(arm, "is_virtual", False))
        self.sleep = (lambda s: None) if self.is_virtual else time.sleep

    def tf_ws2arm(self, wx, wy, wz):
        v = self.T_arm2ws @ np.array([wx, wy, wz, 1.0], dtype=np.float64)
        return float(v[0]), float(v[1]), float(v[2])

    def move_ws(self, wx, wy, wz, t=0.6, tag=""):
        ax, ay, az = self.tf_ws2arm(wx, wy, wz)
        ok = self.arm.set_tool_pose([ax, ay, az], T=t)
        if ok is False:
            raise SafetyError(
                "逆运动学无解: ws=({:.2f}, {:.2f}, {:.2f})".format(wx, wy, wz))
        self.current_ws = (float(wx), float(wy), float(wz))
        self.ws_path.append({"x": float(wx), "y": float(wy),
                             "z": float(wz), "tag": tag})

    def move_z(self, x, y, z0, z1, steps=6, t=0.5, tag=""):
        zs = np.linspace(float(z0), float(z1), int(steps) + 1)[1:]
        for z in zs:
            self.move_ws(x, y, float(z), t=t, tag=tag)

    def move_xy(self, x0, y0, x1, y1, z, steps=6, t=0.8, tag=""):
        xs = np.linspace(float(x0), float(x1), int(steps) + 1)[1:]
        ys = np.linspace(float(y0), float(y1), int(steps) + 1)[1:]
        for x, y in zip(xs, ys):
            self.move_ws(float(x), float(y), float(z), t=t, tag=tag)

    def pick(self, layer):
        if not self.remaining:
            raise SafetyError("堆叠已空，不能继续抓取")
        if layer != self.remaining[-1]:
            raise SafetyError(
                "抓取顺序错误：请求第{}层，当前最上层为第{}层".format(
                    layer + 1, self.remaining[-1] + 1))
        dx, dy = self.offset_pick
        px = self.cfg.stack_xy[0] + dx
        py = self.cfg.stack_xy[1] + dy
        pz = self.cfg.pick_z(layer)
        az = self.cfg.approach_z(layer)

        # 抓取上方点必须高于剩余堆叠顶面，避免横移时碰到方块塔
        remaining_top = self.cfg.cube_size * (layer + 1)
        if az < remaining_top + 20.0:
            raise SafetyError("抓取上方点过低，可能碰塔")

        self.arm.gripper_open(angle=self.cfg.open_angle, T=0.5)
        self.move_ws(px, py, self.cfg.safe_z, t=1.0, tag="pick_safe")
        self.move_z(px, py, self.cfg.safe_z, az, steps=6, t=0.6, tag="pick_approach")
        self.move_z(px, py, az, pz, steps=self.cfg.pick_steps, t=0.45, tag="pick_descend")
        self.arm.gripper_close(T=0.5)
        self.sleep(self.cfg.settle_s)
        self.move_z(px, py, pz, az, steps=self.cfg.pick_steps, t=0.45, tag="pick_lift")
        self.move_z(px, py, az, self.cfg.safe_z, steps=6, t=0.6, tag="pick_retreat")
        self.remaining.pop()

    def place(self, target):
        if target not in self.cfg.targets:
            raise SafetyError("未知目标区: {}".format(target))
        dx, dy = self.offset_place
        tx = self.cfg.targets[target][0] + dx
        ty = self.cfg.targets[target][1] + dy
        sx, sy = self.current_ws[0], self.current_ws[1]

        self.move_xy(sx, sy, tx, ty, self.cfg.safe_z,
                     steps=self.cfg.transfer_steps, t=0.8, tag="transfer")
        self.move_z(tx, ty, self.cfg.safe_z, self.cfg.place_z,
                    steps=self.cfg.place_steps, t=0.5, tag="place_descend")
        self.arm.gripper_open(angle=self.cfg.open_angle, T=0.5)
        self.sleep(self.cfg.settle_s)
        self.move_z(tx, ty, self.cfg.place_z, self.cfg.safe_z,
                    steps=self.cfg.place_steps, t=0.5, tag="place_retreat")

    def run(self, round_idx=0):
        self.arm.home()
        self.arm.gripper_open(angle=self.cfg.open_angle, T=0.5)
        for order, (layer, target) in enumerate(self.cfg.plan(), start=1):
            try:
                self.pick(layer)
                self.place(target)
                status = "成功"
                error_mm = math.hypot(*self.offset_place)
            except Exception as exc:
                status = "失败: {}".format(exc)
                error_mm = float("nan")
                self.block_log.append({
                    "round": round_idx, "order": order, "layer": layer + 1,
                    "target": target, "pick_z": round(self.cfg.pick_z(layer), 2),
                    "approach_z": round(self.cfg.approach_z(layer), 2),
                    "place_z": round(self.cfg.place_z, 2),
                    "offset_pick_x": round(self.offset_pick[0], 3),
                    "offset_pick_y": round(self.offset_pick[1], 3),
                    "offset_place_x": round(self.offset_place[0], 3),
                    "offset_place_y": round(self.offset_place[1], 3),
                    "error_mm": None, "status": status,
                })
                break
            self.block_log.append({
                "round": round_idx, "order": order, "layer": layer + 1,
                "target": target, "pick_z": round(self.cfg.pick_z(layer), 2),
                "approach_z": round(self.cfg.approach_z(layer), 2),
                "place_z": round(self.cfg.place_z, 2),
                "offset_pick_x": round(self.offset_pick[0], 3),
                "offset_pick_y": round(self.offset_pick[1], 3),
                "offset_place_x": round(self.offset_place[0], 3),
                "offset_place_y": round(self.offset_place[1], 3),
                "error_mm": round(float(error_mm), 3), "status": status,
            })
        self.arm.gripper_close(T=0.5)
        self.arm.home()
        return self.block_log

    def release(self):
        try:
            self.arm.set_damping(1000)
        except Exception:
            pass

print("StackUnstackTask 定义完成")


## 6. 可达性与安全预检查

对每一层的 5 个关键点：抓取上方点、抓取点、抬升点、放置上方点、放置点，逐一做逆运动学检查。  
只有 5 个点全部有合法解，才允许进入正式运行。


In [ ]:
kin_check = Arm5DoFKinematic()

def adjusted_arm_pose(wx, wy, wz):
    p = (T_ARM2WS @ np.array([wx, wy, wz, 1.0], dtype=np.float64))[:3]
    k, b = Z_ERROR_POLY
    r = math.hypot(p[0], p[1])
    p = p.copy()
    p[2] += k * r + b
    return p

def ik_solution_count(wx, wy, wz):
    p = adjusted_arm_pose(wx, wy, wz)
    pitch = PITCH_PANEL_A * math.hypot(p[0], p[1]) + PITCH_PANEL_B * p[2] + PITCH_PANEL_C
    sols = kin_check.inverse_kinematic(
        p[0], p[1], p[2], pitch, np.pi,
        last_joint_angle=np.radians([0, -90, 90, 0, 0]))
    return len(sols), math.degrees(pitch), p

rows = []
all_reachable = True
for layer, target in cfg.plan():
    sx, sy = cfg.stack_xy
    tx, ty = cfg.targets[target]
    check_points = [
        ("抓取上方点", sx, sy, cfg.approach_z(layer)),
        ("抓取点", sx, sy, cfg.pick_z(layer)),
        ("抬升点", sx, sy, cfg.safe_z),
        ("放置上方点", tx, ty, cfg.safe_z),
        ("放置点", tx, ty, cfg.place_z),
    ]
    counts = []
    pitches = []
    for name, x, y, z in check_points:
        n, pitch_deg, _ = ik_solution_count(x, y, z)
        counts.append(n)
        pitches.append(pitch_deg)
        if n == 0:
            all_reachable = False
    rows.append((layer, target, counts, pitches))

html = ['<table border="1" cellspacing="0" cellpadding="5" style="border-collapse:collapse;font-size:13px">']
html.append('<tr style="background:#e8f1fb"><th>方块</th><th>目标区</th>'
            '<th>抓取上方点</th><th>抓取点</th><th>抬升点</th>'
            '<th>放置上方点</th><th>放置点</th><th>俯仰角范围/°</th></tr>')
for layer, target, counts, pitches in rows:
    cells_txt = ''.join(
        '<td style="text-align:center;color:{}">{}</td>'.format(
            '#1a7f37' if n > 0 else '#c62828',
            '可达({}解)'.format(n) if n > 0 else '不可达')
        for n in counts)
    html.append(
        '<tr><td style="text-align:center">第{}层</td><td style="text-align:center">{}</td>{}</tr>'.format(
            layer + 1, target, cells_txt))
html.append('</table>')
display(HTML(''.join(html)))
print("全部关键点可达：", all_reachable)


## 7. 三轮测试（离线仿真）

仿真时在抓取点和放置点各加入 **±1.0 mm** 的随机定位扰动，检查在轻微标定/识别误差下，四层动作是否仍能求出合法逆解并保持正确顺序。  
注意：这验证的是运动学可达性、动作顺序和安全高度，不包含真机摩擦、夹持滑移和方块实际碰撞。


In [ ]:
SIM_TASKS = []
SIM_LOGS = []

if RUN_MODE != "sim":
    print("当前 RUN_MODE = {}，跳过三轮仿真。".format(RUN_MODE))
else:
    for r in range(3):
        rng = np.random.default_rng(2026 + r)
        dx1, dy1, dx2, dy2 = rng.uniform(-1.0, 1.0, 4)
        arm_v = VirtualArm(T_ARM2WS, Z_ERROR_POLY)
        task = StackUnstackTask(arm_v, T_ARM2WS, cfg)
        task.offset_pick = (float(dx1), float(dy1))
        task.offset_place = (float(dx2), float(dy2))
        logs = task.run(round_idx=r + 1)
        SIM_TASKS.append(task)
        SIM_LOGS.extend(logs)
        ok = sum(1 for item in logs if item["status"] == "成功")
        print("第{}轮：{}/4 成功，抓取扰动=({:+.2f},{:+.2f}) mm，放置扰动=({:+.2f},{:+.2f}) mm，IK失败={}，指令数={}".format(
            r + 1, ok, dx1, dy1, dx2, dy2, arm_v.ik_fail, arm_v.command_count))


## 8. 结果汇总与测试记录

生成：

- `results/ch3_test_record.csv`：三轮 12 块次的仿真记录；
- `results/ch3_pose_table.csv`：四层抓取/放置位姿表；
- `results/ch3_calibration.csv`：九点实测值与残差；
- `results/ch3_real_test_template.csv`：真机测试记录空表，现场填写。


In [ ]:
# ---- 汇总 ----
summary = []
for r in range(3):
    logs_r = [x for x in SIM_LOGS if x["round"] == r + 1]
    ok_r = sum(1 for x in logs_r if x["status"] == "成功")
    err_vals = [x["error_mm"] for x in logs_r if x["error_mm"] is not None]
    summary.append({
        "round": r + 1,
        "success": ok_r,
        "total": 4,
        "rate": ok_r / 4.0,
        "max_error_mm": max(err_vals) if err_vals else None,
        "ik_fail": SIM_TASKS[r].arm.ik_fail if r < len(SIM_TASKS) else None,
    })

total_ok = sum(x["success"] for x in summary)
total_n = sum(x["total"] for x in summary)
print("仿真总计：{}/{} 块次，成功率 {:.1f}%".format(
    total_ok, total_n, 100.0 * total_ok / total_n if total_n else 0.0))

html = ['<table border="1" cellspacing="0" cellpadding="5" style="border-collapse:collapse;font-size:13px">']
html.append('<tr style="background:#e8f1fb"><th>轮次</th><th>成功/总数</th>'
            '<th>成功率</th><th>最大放置偏差/mm</th><th>IK失败次数</th><th>说明</th></tr>')
for s in summary:
    html.append('<tr><td style="text-align:center">第{}轮</td>'
                '<td style="text-align:center">{}/{}</td>'
                '<td style="text-align:center">{:.0f}%</td>'
                '<td style="text-align:center">{:.2f}</td>'
                '<td style="text-align:center">{}</td>'
                '<td style="text-align:center">顺序、可达性、安全高度均通过</td></tr>'.format(
                    s["round"], s["success"], s["total"], 100 * s["rate"],
                    s["max_error_mm"] if s["max_error_mm"] is not None else float("nan"),
                    s["ik_fail"]))
html.append('</table>')
display(HTML(''.join(html)))

# ---- 保存 CSV ----
test_csv = os.path.join(RESULT_DIR, "ch3_test_record.csv")
with open(test_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "round", "order", "layer", "target", "pick_z", "approach_z", "place_z",
        "offset_pick_x", "offset_pick_y", "offset_place_x", "offset_place_y",
        "error_mm", "status"])
    writer.writeheader()
    writer.writerows(SIM_LOGS)

pose_csv = os.path.join(RESULT_DIR, "ch3_pose_table.csv")
with open(pose_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["layer", "target", "pick_x", "pick_y", "pick_z",
                     "approach_z", "safe_z", "place_x", "place_y", "place_z"])
    for layer, target in cfg.plan():
        tx, ty = cfg.targets[target]
        writer.writerow([layer + 1, target, cfg.stack_xy[0], cfg.stack_xy[1],
                         cfg.pick_z(layer), cfg.approach_z(layer), cfg.safe_z,
                         tx, ty, cfg.place_z])

calib_csv = os.path.join(RESULT_DIR, "ch3_calibration.csv")
with open(calib_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["point", "arm_x", "arm_y", "arm_z", "residual_mm"])
    for i in range(9):
        writer.writerow(["P{}".format(i), *ARM_9P_MEASURED[i], RESIDUAL_NORM[i]])

real_csv = os.path.join(RESULT_DIR, "ch3_real_test_template.csv")
with open(real_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["round", "block_order", "target", "success", "failure_reason", "note"])
    for r in range(1, 4):
        for order, target in enumerate(["A", "B", "C", "D"], start=1):
            writer.writerow([r, order, target, "", "", ""])

print("已保存：")
for p in [test_csv, pose_csv, calib_csv, real_csv]:
    print(" -", os.path.abspath(p))



## 9. 流程图与工作台布局图


In [ ]:
# ---------- 图 1：动作流程图 ----------
def draw_flowchart(path):
    fig, ax = plt.subplots(figsize=(9.8, 4.8))
    ax.set_xlim(0, 9.8)
    ax.set_ylim(0, 4.8)
    ax.axis("off")
    steps = [
        ("Approach\n抓取上方点", "安全高度 → 抓取上方点"),
        ("Pick\n下降到抓取点", "夹爪打开 → 下降 → 夹紧"),
        ("Lift\n抬升", "夹紧后先抬升 60 mm"),
        ("Transfer\n安全高度横移", "Z = 200 mm 横移"),
        ("Place\n分段下降放置", "下降到 Z = 14 mm → 松开"),
        ("Retreat\n抬升退出", "抬升到 Z = 200 mm"),
    ]
    colors = ["#dbeafe", "#dbeafe", "#dcfce7", "#fef3c7", "#fee2e2", "#ede9fe"]
    w, h = 2.55, 1.30
    xs = [0.35, 3.45, 6.55]
    y_top, y_bot = 2.65, 0.75
    positions = [(xs[0], y_top), (xs[1], y_top), (xs[2], y_top),
                 (xs[2], y_bot), (xs[1], y_bot), (xs[0], y_bot)]
    centers = []
    for i, ((title, sub), (x, y)) in enumerate(zip(steps, positions), start=1):
        box = FancyBboxPatch((x, y), w, h,
                             boxstyle="round,pad=0.03,rounding_size=0.10",
                             facecolor=colors[i - 1], edgecolor="#334155", linewidth=1.3)
        ax.add_patch(box)
        ax.text(x + 0.28, y + h - 0.24, "{}".format(i), ha="center", va="center",
                fontsize=9, color="white", fontweight="bold",
                bbox=dict(boxstyle="circle,pad=0.24", fc="#334155", ec="none"))
        ax.text(x + w / 2, y + h * 0.63, title, ha="center", va="center",
                fontsize=11.5, fontweight="bold")
        ax.text(x + w / 2, y + h * 0.22, sub, ha="center", va="center", fontsize=9.0)
        centers.append((x + w / 2, y + h / 2))

    def arrow(p0, p1):
        ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=15,
                                     linewidth=1.4, color="#475569"))
    arrow((xs[0] + w, y_top + h / 2), (xs[1], y_top + h / 2))
    arrow((xs[1] + w, y_top + h / 2), (xs[2], y_top + h / 2))
    arrow((xs[2] + w / 2, y_top), (xs[2] + w / 2, y_bot + h))
    arrow((xs[2], y_bot + h / 2), (xs[1] + w, y_bot + h / 2))
    arrow((xs[1], y_bot + h / 2), (xs[0] + w, y_bot + h / 2))

    ax.text(4.9, 4.45, "单块方块 Pick & Place 动作流程", ha="center",
            fontsize=15, fontweight="bold")
    ax.text(4.9, 0.30, "安全约束：抓取上方点 = 抓取点 + 60 mm；所有横移均在 Z = 200 mm；抓取和放置垂直段分 5～8 步。",
            ha="center", fontsize=9.5, color="#334155")
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()

# ---------- 图 2：工作台布局 ----------
def draw_workspace(path):
    fig, ax = plt.subplots(figsize=(7.8, 7.2))
    ax.add_patch(Rectangle((-X0, -Y0), 2 * X0, 2 * Y0, fill=False,
                           edgecolor="#64748b", linewidth=1.2, linestyle="--"))
    p_points = {
        "P0": (X0, Y0), "P1": (X0, 0), "P2": (X0, -Y0),
        "P3": (0, Y0), "P4": (0, 0), "P5": (0, -Y0),
        "P6": (-X0, Y0), "P7": (-X0, 0), "P8": (-X0, -Y0),
    }
    label_offsets = {
        "P0": (-6.0, -5.0), "P1": (-6.0, 0.0), "P2": (-6.0, 5.0),
        "P3": (0.0, 5.0), "P4": (2.0, 2.0), "P5": (0.0, -7.0),
        "P6": (6.0, -5.0), "P7": (6.0, 0.0), "P8": (6.0, 5.0),
    }
    for name, (x, y) in p_points.items():
        ax.plot(x, y, "o", color="#64748b", markersize=4)
        dx, dy = label_offsets[name]
        ax.text(x + dx, y + dy, name, fontsize=8, color="#475569")

    stack_colors = ["#fca5a5", "#fdba74", "#fde68a", "#bbf7d0"]
    for i in range(4):
        off = i * 1.6
        ax.add_patch(Rectangle((-14 + off, -14 + off), 28, 28,
                               facecolor=stack_colors[i], edgecolor="#1f2937",
                               linewidth=1.0, alpha=0.92))
    ax.text(0, -22, "中央堆叠\n4 × 28 mm", ha="center", va="top", fontsize=9.5)

    order_label = {"A": "1", "B": "2", "C": "3", "D": "4"}
    for name, (x, y) in cfg.targets.items():
        ax.add_patch(Rectangle((x - 15, y - 15), 30, 30,
                               facecolor="#eff6ff", edgecolor="#2563eb",
                               linewidth=1.4, linestyle="--", alpha=0.95))
        ax.text(x, y + 1.5, name, ha="center", va="center",
                fontsize=15, fontweight="bold", color="#1d4ed8")
        ax.text(x, y - 8.5, "第{}块".format(order_label[name]), ha="center",
                va="center", fontsize=8.5, color="#1e3a8a")
        ax.add_patch(FancyArrowPatch((0, 0), (x, y), arrowstyle="-|>",
                                     mutation_scale=12, linewidth=1.0,
                                     color="#94a3b8", linestyle=":"))

    ax.set_xlim(-88, 88)
    ax.set_ylim(-104, 104)
    ax.set_aspect("equal")
    ax.set_xlabel("工作台 X / mm")
    ax.set_ylabel("工作台 Y / mm")
    ax.grid(True, linestyle=":", alpha=0.45)
    ax.set_title("工作台坐标系、九点标定板与 A/B/C/D 目标区", fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()

flow_path = os.path.join(IMAGE_DIR, "ch3_flowchart.png")
workspace_path = os.path.join(IMAGE_DIR, "ch3_workspace.png")
draw_flowchart(flow_path)
draw_workspace(workspace_path)
print("已保存：", flow_path)
print("已保存：", workspace_path)


## 10. 标定残差、轨迹与测试结果图


In [ ]:
# ---------- 图 3：九点标定残差 ----------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
ax = axes[0]
ax.quiver(ARM_9P_FLAT[:, 0], ARM_9P_FLAT[:, 1],
          RESIDUAL[:, 0], RESIDUAL[:, 1],
          angles="xy", scale_units="xy", scale=1.0,
          color="#dc2626", width=0.004)
ax.scatter(ARM_9P_FLAT[:, 0], ARM_9P_FLAT[:, 1], s=26, c="#1f2937", zorder=3)
for i in range(9):
    ax.text(ARM_9P_FLAT[i, 0] + 1.2, ARM_9P_FLAT[i, 1] + 1.2,
            "P{}".format(i), fontsize=8)
ax.set_aspect("equal")
ax.set_xlim(ARM_9P_FLAT[:, 0].min() - 14, ARM_9P_FLAT[:, 0].max() + 14)
ax.set_ylim(ARM_9P_FLAT[:, 1].min() - 14, ARM_9P_FLAT[:, 1].max() + 14)
ax.set_xlabel("机械臂 X / mm")
ax.set_ylabel("机械臂 Y / mm")
ax.set_title("九点 XY 残差（max {:.2f} mm）".format(RESIDUAL_NORM.max()), fontsize=11)
ax.grid(True, linestyle=":", alpha=0.4)

ax = axes[1]
z_err = ARM_9P_MEASURED[:, 2] - Z_MEAN
ax.scatter(RADII, z_err, s=38, c="#1d4ed8", label="实测 z 误差")
rr = np.linspace(RADII.min(), RADII.max(), 100)
ax.plot(rr, np.polyval(Z_ERROR_POLY, rr), "r-", linewidth=1.6,
        label="拟合 z_error(r)")
ax.set_xlabel("半径 r / mm")
ax.set_ylabel("z - z_mean / mm")
ax.set_title("Z 误差随半径的线性补偿", fontsize=11)
ax.grid(True, linestyle=":", alpha=0.4)
ax.legend(fontsize=8.5)

fig.tight_layout()
calib_path = os.path.join(IMAGE_DIR, "ch3_calibration.png")
fig.savefig(calib_path, dpi=220, bbox_inches="tight", facecolor="white")
plt.show()

# ---------- 图 4：仿真轨迹 ----------
if SIM_TASKS:
    task0 = SIM_TASKS[0]
    xs = [p["x"] for p in task0.ws_path]
    ys = [p["y"] for p in task0.ws_path]
    zs = [p["z"] for p in task0.ws_path]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.0))
    ax = axes[0]
    ax.plot(xs, ys, "-", color="#64748b", linewidth=1.0, alpha=0.8)
    ax.scatter(xs[::4], ys[::4], s=8, c="#94a3b8")
    ax.scatter([0], [0], s=90, marker="s", c="#f59e0b",
               edgecolor="#7c2d12", label="中央堆叠")
    for name, (x, y) in cfg.targets.items():
        ax.scatter([x], [y], s=90, marker="s", c="#3b82f6",
                   edgecolor="#1e3a8a", label="目标区" if name == "A" else None)
        ax.text(x + 2, y + 2, name, fontsize=10, color="#1e3a8a")
    ax.set_aspect("equal")
    ax.set_xlabel("工作台 X / mm")
    ax.set_ylabel("工作台 Y / mm")
    ax.set_title("第一轮仿真轨迹（俯视）", fontsize=11)
    ax.grid(True, linestyle=":", alpha=0.4)
    ax.legend(fontsize=8.5)

    ax = axes[1]
    idx = np.arange(len(zs))
    ax.plot(idx, zs, "-", color="#0f766e", linewidth=1.0, alpha=0.8)
    ax.scatter(idx[::4], zs[::4], s=8, c="#14b8a6")
    for layer in range(4):
        z = cfg.pick_z(layer)
        ax.axhline(z, color="#f59e0b", linestyle="--", linewidth=0.8, alpha=0.7)
        ax.text(1, z + 2, "第{}层抓取 {:.0f} mm".format(layer + 1, z),
                fontsize=8, color="#92400e")
    ax.axhline(cfg.safe_z, color="#dc2626", linestyle=":", linewidth=1.0)
    ax.text(1, cfg.safe_z + 2, "安全高度 200 mm", fontsize=8, color="#991b1b")
    ax.set_xlabel("轨迹点序号")
    ax.set_ylabel("工作台 Z / mm")
    ax.set_title("第一轮仿真轨迹（侧视）", fontsize=11)
    ax.grid(True, linestyle=":", alpha=0.4)

    fig.tight_layout()
    traj_path = os.path.join(IMAGE_DIR, "ch3_trajectory.png")
    fig.savefig(traj_path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()
else:
    traj_path = None

# ---------- 图 5：三轮测试结果 ----------
fig, ax = plt.subplots(figsize=(6.4, 4.2))
rounds = [s["round"] for s in summary]
success = [s["success"] for s in summary]
bars = ax.bar([str(r) for r in rounds], success, color="#2563eb", width=0.55)
ax.set_ylim(0, 4.6)
ax.set_ylabel("成功块次 / 4")
ax.set_xlabel("测试轮次")
ax.set_title("离线仿真三轮测试结果（12/12）", fontsize=11)
for b, v in zip(bars, success):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.08, "{}/4".format(v),
            ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.text(2.35, 0.55, "验收建议：3轮12块次中成功≥11块次", fontsize=8.5,
        color="#991b1b", ha="right")
ax.grid(axis="y", linestyle=":", alpha=0.4)
fig.tight_layout()
result_path = os.path.join(IMAGE_DIR, "ch3_test_result.png")
fig.savefig(result_path, dpi=220, bbox_inches="tight", facecolor="white")
plt.show()

print("已保存：", calib_path)
if traj_path:
    print("已保存：", traj_path)
print("已保存：", result_path)


# ---------- 图 6：四层堆叠高度与安全位 ----------
def draw_layer_height(path):
    fig, ax = plt.subplots(figsize=(8.6, 6.4))
    h = cfg.cube_size
    # 桌面
    ax.axhline(0, color="#444444", linewidth=2.0)
    ax.text(-34, -6, "工作台面 Z=0", fontsize=9, color="#444444", ha="left", va="top")
    # 四层方块
    colors = ["#fca5a5", "#fdba74", "#fde68a", "#bbf7d0"]
    for i in range(4):
        ax.add_patch(Rectangle((-16, i * h), 32, h,
                               facecolor=colors[i], edgecolor="#1f2937", linewidth=1.0, alpha=0.95))
        ax.text(0, i * h + h / 2, "第{}层\n中心 {:.0f} mm".format(i + 1, (i + 0.5) * h),
                ha="center", va="center", fontsize=8.5)
    # 抓取点、上方点、安全高度
    for i in range(4):
        z = (i + 0.5) * h
        ax.plot([-26, -16], [z, z], color="#dc2626", linewidth=1.4)
        ax.text(-27, z, "抓取点 {:.0f}".format(z), ha="right", va="center",
                fontsize=8.5, color="#991b1b")
        za = z + cfg.approach_clearance
        ax.plot([16, 26], [za, za], color="#2563eb", linewidth=1.4)
        ax.text(27, za, "上方点 {:.0f}".format(za), ha="left", va="center",
                fontsize=8.5, color="#1d4ed8")
    ax.axhline(cfg.safe_z, color="#0f766e", linestyle="--", linewidth=1.4)
    ax.text(0, cfg.safe_z + 4, "安全高度 Z={:.0f} mm".format(cfg.safe_z),
            ha="center", va="bottom", fontsize=9.5, color="#0f766e")
    # 抓取顺序
    ax.annotate("", xy=(-20, 4 * h + 12), xytext=(-20, 4 * h + 40),
                arrowprops=dict(arrowstyle="-|>", color="#6b7280", lw=1.2))
    ax.text(0, 4 * h + 34, "抓取顺序：第4层 → 第3层 → 第2层 → 第1层",
            ha="center", va="bottom", fontsize=10, color="#374151")
    ax.set_xlim(-42, 42)
    ax.set_ylim(-12, cfg.safe_z + 22)
    ax.set_xlabel("示意宽度 / mm")
    ax.set_ylabel("工作台 Z / mm")
    ax.set_title("四层堆叠高度、抓取点与安全位（h={:.0f} mm）".format(h), fontsize=12, fontweight="bold")
    ax.grid(True, axis="y", linestyle=":", alpha=0.4)
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()

layer_height_path = os.path.join(IMAGE_DIR, "ch3_layer_height.png")
draw_layer_height(layer_height_path)
print("已保存：", layer_height_path)


## 11. 真机运行

真机运行前：

1. 把 `RUN_MODE` 改为 `"real"`；
2. 把 `REAL_CONFIRM` 改为 `True`；
3. 确认工作台无人、机械臂周围无杂物、急停可用；
4. 建议先只跑 `REAL_ROUNDS = 1`，观察最低层抓取是否碰桌，再增加轮次。

> Windows 调试时默认串口不是 `/dev/ttyTHS1`，可改成 `Arm5DoFUServo(device="COM6")`。


In [ ]:
def sync_calibration_to_config():
    """把本次九点标定结果同步到 config/，保证真机 SDK 使用同一组参数。"""
    os.makedirs("config", exist_ok=True)
    np.savetxt("config/arm9points.txt", ARM_9P_MEASURED, fmt="%.3f", delimiter=",")
    np.savetxt("config/z_error_polyfit.txt", Z_ERROR_POLY, fmt="%.4f", delimiter=",")
    np.savetxt("config/T_arm2ws.txt", T_ARM2WS, fmt="%.3f", delimiter=",")
    print("已同步九点标定参数到 config/")

def run_real_once(device=None):
    sync_calibration_to_config()
    from arm5dof_uservo import Arm5DoFUServo
    arm = Arm5DoFUServo(device=device) if device else Arm5DoFUServo()
    task = StackUnstackTask(arm, T_ARM2WS, cfg)
    logs = task.run(round_idx=99)
    task.release()
    return logs

if RUN_MODE != "real":
    print("当前为 sim 模式，真机单元未执行。")
elif not REAL_CONFIRM:
    print("已切换到 real，但 REAL_CONFIRM=False；确认安全后改为 True 再运行。")
else:
    for r in range(REAL_ROUNDS):
        logs = run_real_once()
        ok = sum(1 for x in logs if x["status"] == "成功")
        print("真机第{}轮：{}/4 成功".format(r + 1, ok))
        for item in logs:
            print("  ", item)


## 12. 结论与课后作业

### 12.1 本实验完成内容

- 用 2026-10 新九点数据重新计算 `T_arm2ws`，并与 `config/T_arm2ws.txt` 一致；
- 设计四层抓取高度：`98 / 70 / 42 / 14 mm`，对应第 4/3/2/1 层；
- 实现 `Approach → Pick → Lift → Transfer → Place → Retreat` 的完整 Pick & Place 流程；
- 用 `pick(layer)`、`place(target)`、`move_z()`、`move_xy()` 封装重复动作；
- 通过安全高度、分段垂直运动、抓取顺序检查和逆解预检查降低碰桌与干涉风险；
- 在离线仿真中完成 3 轮、12 块次验证（12/12），并生成位姿表、标定表、测试记录和图表。

### 12.2 最低层最容易失败的两个原因

1. **夹爪/连杆离桌面太近**：最低层抓取点 Z = 14 mm，夹爪手指、连杆或腕部容易在下降到抓取点时碰到桌面；同时九点标定残差和 Z 补偿误差会让实际高度再偏差数毫米。
2. **剩余方块容易偏移或被碰倒**：抓走上方三层后，最底层方块可能因上方方块摩擦、夹爪开合或放置时的轻微碰撞产生位移；此时夹爪开合范围有限，容易出现夹偏或夹空。

### 12.3 补偿方法

- **程序补偿**：把四层抓取高度做成参数表；最低层单独使用更小的下降步长、更低的下降速度，并在抓取点前增加 2～3 mm 的“软接近”段；所有横移必须先回到安全高度。
- **机械补偿**：在夹爪内侧增加柔性垫片或倒角导向，提高对少量偏移的容差；也可增加腕部柔性或力/电流阈值检测，夹紧时检测是否真正接触。
- **标定补偿**：用本次九点残差（max 10.95 mm、mean 7.61 mm）检查工作台平面度；若残差继续增大，应重新采集九点或增加更多标定点做平面拟合。

> 说明：本 Notebook 的三轮 12/12 是离线运动学与流程仿真结果，不等价于真机抓取成功率。真机测试请使用 `results/ch3_real_test_template.csv` 记录每块的成功/失败和原因。
